# 5장 1강 · VLM의 개념과 활용

실습파일

사진을 보지 못한 사람에게 장면을 한 문장으로 설명하는 일, 얼룩말 수를 입력하는 일, 지도에 표지판의 위치와 글자를 등록하는 일에는 서로 다른 출력이 필요합니다. 같은 이미지에서 어떤 정보를 어떤 모델로 얻을지 정하고, 실제로 생성한 답을 검수해 봅시다.

## 실습 안내

| 구간 | 다룰 질문 | 완성할 결과 |
|---|---|---|
| 필수 1 | 사진을 설명하는 모델과 대상의 위치를 찾는 모델은 무엇을 돌려줄까? | 생성 캡션, 원본·주석·예측 상자, 겹침 정도 |
| 필수 2 | 답변 형식을 지정하면 마릿수까지 더 정확해질까? | 같은 사진에 두 질문을 적용한 비교표, 오류 사례 |
| 과제 1 | 글자를 읽기 전에 표지판부터 찾으면 무엇이 달라질까? | 좌표와 판독 결과, 무문자 대조 실험, 검수 판정 |

**이 수업에서 쓰는 용어**  VLM(Vision-Language Model)은 사진의 시각 정보와 질문의 언어 정보를 함께 사용해 답을 만드는 모델입니다. 캡션 생성은 사진을 소개하는 문장을 만드는 작업이고, VQA(시각적 질문 응답)는 사진을 근거로 특정 질문에 답하는 작업입니다. ‘몇 마리인가?’에 답하기와 ‘표지판에 뭐라고 쓰였는가?’에 답하기가 이 수업의 VQA입니다. 후자는 이미지 속 글자를 읽는 OCR 과제이기도 합니다.

**준비물과 실행 방법**  강의용 `coco_sample.zip` 또는 `coco_dataset.zip`을 노트북과 같은 폴더에 둡니다. 압축을 풀었다면 `coco_dataset` 폴더도 지원합니다. COCO 사진 번호 11760(얼룩말), 15440(도로), 별도로 고른 얼룩말 사진 9장을 씁니다. COCO 주석은 모델에 제공하지 않고 결과를 확인할 때 씁니다. 평가 표본을 구성할 때에만 주석의 개수를 참조합니다.

공통 준비 ①~③을 실행하면 필수 1, 필수 2, 과제 1을 각각 시작할 수 있습니다. 각 구간 안에서는 번호순으로 진행하세요. 정답 파일에는 실행 출력과 사진이 저장되어 있습니다. 새로 실행할 때는 첫 모델 다운로드에 인터넷이 필요하며 API 키는 필요하지 않습니다. 검증 환경은 Python 3.12, PyTorch 2.6.0(CPU), torchvision 0.21.0, transformers 4.53.3입니다. 새 Jupyter 환경에서는 `%pip install torch==2.6.0 torchvision==0.21.0 transformers==4.53.3 pandas pillow` 실행 후 커널을 재시작하세요.

**실험 조건**  SmolVLM-256M-Instruct는 이미지와 질문으로 문장을 생성하고, COCO 가중치를 사용한 SSDLite는 클래스·상자·점수를 예측합니다. 두 모델 모두 사전학습 가중치를 고정하고 실제 사진에 대해 추론합니다. 생성 조건을 일정하게 유지하기 위해 영어로 질문하고 샘플링을 끕니다. 기존 자료의 한국어 응답 A~D는 비교용 기록으로 표시합니다. 학생은 해석과 답안을 한국어로 작성하세요.

### 공통 준비 ① · 사진과 두 종류의 참조 자료 읽기

`objects.csv`의 한 행은 사람이 표시한 객체 주석입니다. `captions.csv`에는 한 사진을 설명한 문장이 여러 개 있습니다. 두 표를 행 번호로 연결하면 사진이 뒤섞일 수 있어 `image_id`로 연결합니다. `instances`는 그 사진에서 지정한 클래스의 개별 주석만 고릅니다. `iscrowd=1`은 여러 개체를 한꺼번에 표시한 영역이어서 한 마리처럼 세기 어렵습니다. 그림 아래 ‘GT’는 사람이 만든 참조, ‘P’는 모델의 예측을 뜻합니다.

In [1]:
from pathlib import Path
from zipfile import ZipFile
from io import BytesIO
from math import floor, ceil
import re
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageOps
from IPython.display import display

locations = [Path("coco_dataset.zip")]
DATA = next((p for p in locations if p.exists()), None)
if DATA is None:
    raise FileNotFoundError("강의용 COCO ZIP 또는 압축 해제 폴더를 locations에 지정하세요.")
if DATA.is_dir():
    root = next(DATA.rglob("objects.csv")).parent
    def present(name):
        return (root / name).is_file()
    def read_bytes(name):
        return (root / name).read_bytes()
else:
    with ZipFile(DATA) as z:
        prefix = next(n for n in z.namelist() if n.split("/")[-1]=="objects.csv")[:-len("objects.csv")]
        members = set(z.namelist())
    def present(name):
        return prefix+name in members
    def read_bytes(name):
        with ZipFile(DATA) as z:
            return z.read(prefix+name)

objects = pd.read_csv(BytesIO(read_bytes("objects.csv")))
captions = pd.read_csv(BytesIO(read_bytes("captions.csv")))
paths = captions.drop_duplicates("image_id").set_index("image_id").image_path
paths = paths[paths.map(present)]
def photo(key):
    with Image.open(BytesIO(read_bytes(paths.at[key]))) as im:
        return im.convert("RGB")
def instances(key, kind):
    return objects.loc[(objects.image_id==key)&(objects.class_name==kind)&(objects.iscrowd==0)].copy()
coords = ["xmin","ymin","xmax","ymax"]
def painted(im, boxes, color, label):
    canvas = im.copy()
    draw = ImageDraw.Draw(canvas)
    for i,box in enumerate(boxes):
        draw.rectangle(tuple(box),outline=color,width=3)
        draw.text((box[0]+3,box[1]+3),f"{label}{i+1}",fill=color,stroke_width=1,stroke_fill="black")
    return canvas
def grid(items):
    cols = min(3,len(items))
    canvas = Image.new("RGB",(cols*240, ((len(items)+cols-1)//cols)*196),"white")
    draw = ImageDraw.Draw(canvas)
    for i,(label,im) in enumerate(items):
        thumb = ImageOps.contain(im,(234,165))
        x,y = i%cols*240,i//cols*196
        canvas.paste(thumb,(x+(234-thumb.width)//2,y))
        draw.text((x+3,y+168),label,fill="black")
    display(canvas)
print("고정 예제", {key:photo(key).size for key in [11760,15440]})

고정 예제 {11760: (640, 425), 15440: (404, 640)}


### 공통 준비 ② · 사진과 질문을 실제 모델에 넣는 방법

① `encode`는 사용자 메시지에 이미지 한 자리와 질문을 넣습니다. 프로세서는 모델이 읽을 픽셀 배열, 유효한 위치를 표시하는 마스크, 질문을 숫자로 바꾼 토큰 ID를 만듭니다. ② `ask`는 이 입력으로 새 토큰을 최대 48개 생성합니다. 전체 결과에서 입력 길이만큼 앞쪽을 잘라내고 새 답만 글자로 복원합니다. ③ `detect`는 원본 RGB 이미지를 탐지 모델의 가중치에 맞게 전처리하고 상자 좌표를 **원본 사진 기준**으로 돌려줍니다.

CPU 한 대에서 진행할 수 있도록 VLM은 이미지를 여러 조각으로 나누지 않고 긴 변 512 설정을 사용합니다. 원본·잘라낸 영역·대조 입력에 같은 설정을 적용합니다. `eval()`은 모델을 평가 모드로 바꾸고 `inference_mode()`는 추론에 필요 없는 기울기 기록을 생략합니다.

In [2]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText
from torchvision.models.detection import ssdlite320_mobilenet_v3_large, SSDLite320_MobileNet_V3_Large_Weights

torch.set_num_threads(2)
device = "cpu"
model_id = "HuggingFaceTB/SmolVLM-256M-Instruct"
revision = "d039ab0d4b9311b617e8ffec1d06c8464364aeb3"
processor = AutoProcessor.from_pretrained(model_id,revision=revision)
vlm = AutoModelForImageTextToText.from_pretrained(
    model_id,revision=revision,torch_dtype=torch.float32,attn_implementation="eager"
).to(device).eval()
# 원본 비율을 유지하고 긴 변을 512로 맞춰 한 장의 화면으로 처리합니다.
processor.image_processor.do_image_splitting = False
processor.image_processor.size = {"longest_edge":512}
det_weights = SSDLite320_MobileNet_V3_Large_Weights.DEFAULT
detector = ssdlite320_mobilenet_v3_large(weights=det_weights).to(device).eval()

def encode(im, question):
    chat = [{"role":"user","content":[{"type":"image"},{"type":"text","text":question}]}]
    text = processor.apply_chat_template(chat,add_generation_prompt=True,tokenize=False)
    return processor(text=text,images=[im],return_tensors="pt").to(device)

@torch.inference_mode()
def ask(im, question):
    inputs = encode(im,question)
    tokens = vlm.generate(**inputs,max_new_tokens=48,do_sample=False)
    new_tokens = tokens[:,inputs.input_ids.shape[1]:]
    return processor.batch_decode(new_tokens,skip_special_tokens=True)[0].strip()

@torch.inference_mode()
def detect(im):
    out = detector([det_weights.transforms()(im).to(device)])[0]
    table = pd.DataFrame(out["boxes"].cpu().numpy(),columns=coords)
    table["class_name"] = [det_weights.meta["categories"][i] for i in out["labels"].tolist()]
    table["score"] = out["scores"].cpu().numpy()
    return table

caption_q = "Describe this picture in one sentence."
open_q = "How many zebras can you see?"
strict_q = "Count only visible zebras. Reply with a single digit and no other words."
ocr_q = "Read the word on the traffic sign. Reply with only the word in uppercase. If unreadable, reply UNKNOWN."
print("사용 모델",model_id,"/",det_weights.name,"| 추론 장치",device)

ModuleNotFoundError: No module named 'transformers'

### 공통 준비 ③ · 위치·형식·내용을 따로 확인하는 도구

`iou`는 두 상자가 겹치는 넓이를 두 상자가 차지하는 전체 넓이로 나눕니다. `crop_box`는 실수 좌표의 왼쪽·위 경계를 내리고 오른쪽·아래 경계를 올린 다음 이미지 안에 맞춥니다. `number_check`는 답의 형식과 마릿수를 따로 기록합니다. 원문은 그대로 보존합니다. 이 실습의 형식 기준은 양끝 공백을 제외하고 0~9 숫자 한 글자입니다. 내용 검사에서는 문장 속 영문 수사나 숫자를 찾되, 서로 다른 후보가 둘 이상이거나 없으면 보류합니다. 부정문이나 복잡한 문장의 뜻은 원문과 사진으로 확인하세요.

In [ ]:
def iou(box1,box2):
    x1,y1,x2,y2 = box1
    u1,v1,u2,v2 = box2
    inter = max(0,min(x2,u2)-max(x1,u1))*max(0,min(y2,v2)-max(y1,v1))
    area1,area2 = (x2-x1)*(y2-y1),(u2-u1)*(v2-v1)
    return float(inter/(area1+area2-inter))

def crop_box(im,box):
    x1,y1,x2,y2 = box
    limits = (max(0,floor(x1)),max(0,floor(y1)),min(im.width,ceil(x2)),min(im.height,ceil(y2)))
    return limits, im.crop(limits)

number_words = "zero one two three four five six seven eight nine ten".split()
def read_count(raw):
    # 정수 또는 영어 수사만 후보로 뽑습니다. 뜻이 모호한 문장은 사람이 확인합니다.
    if re.fullmatch(r"there are no zebras(?: present)?(?: in the image)?[.!]?",raw.strip().lower()):
        return 0
    tokens = re.findall(r"\b(?:[0-9]+|"+"|".join(number_words)+r")\b",raw.lower())
    values = {int(t) if t.isdigit() else number_words.index(t) for t in tokens}
    return next(iter(values)) if len(values)==1 else None
def number_check(raw,truth):
    digit_only = re.fullmatch(r"[0-9]",raw.strip()) is not None
    value = read_count(raw)
    return {"raw":raw,"format_ok":digit_only,"value":value,
            "count_ok":value==truth,"both_ok":digit_only and value==truth}

# 첨부한 기존 자료의 한국어 질문·응답 기록. 이 표와 현재 모델 출력은 별도로 표시합니다.
previous = pd.DataFrame([
    ("A",11760,"원본","이 이미지를 설명해 주세요.","줄무늬가 있는 얼룩말들이 나무 앞에 나란히 서서 이쪽을 바라보고 있습니다."),
    ("B",11760,"원본","사진에 보이는 얼룩말은 몇 마리인가요? 숫자만 쓰고, 가려져 보이지 않는 동물은 추측하지 마세요.","3"),
    ("C",15440,"원본","사진에서 가장 눈에 띄는 대상을 배경과 함께 한 문장으로 설명해 주세요.","도로 앞쪽에 STOP이라고 적힌 붉은 표지판이 서 있고, 뒤쪽에는 차량과 건물이 보입니다."),
    ("D",15440,"표지판 영역","표지판에 보이는 영문만 대문자로 적어 주세요. 읽을 수 없으면 판독 불가라고 답하세요.","STOP")
],columns=["기록","image_id","범위","질문","응답"]).set_index("기록")

## 필수 1 · 한 사진에서 객체 위치와 장면 설명 대조하기

대상 수를 사람이 검산할 기준부터 만든 뒤, 실제 VLM 캡션과 탐지기 좌표가 어떤 질문에 답하는지 비교합니다.

### 1-1. 세 마리를 세었다는 근거 남기기

사진 11760에서 개별 얼룩말 주석만 골라 왼쪽부터 정렬하세요. 원본과 참조 상자를 나란히 출력하고, 첫 상자의 너비·높이를 좌표에서 읽으세요. 이 표를 모델의 예측 결과로 사용해도 되는지 설명하세요.

**풀이 단서**  
instances와 coords를 이용하세요. 상자 너비는 xmax−xmin, 높이는 ymax−ymin입니다.

In [ ]:
# 여기에 풀이를 작성하세요.
# im = photo(11760)
# gt = instances(...).sort_values(...)
# grid와 painted로 원본과 주석을 비교하세요.

**나의 해석**  
출력에서 근거를 한 가지 이상 골라 작성하세요.

✎

### 1-2. 픽셀과 질문을 전처리해 새 캡션 생성하기

원본과 장면 설명 질문을 encode에 넣어 이미지·토큰·마스크의 모양을 출력하세요. 이어 ask를 실행해 영문 캡션을 생성하고, 입력 토큰 수와 답변 토큰 수가 어떻게 구분되는지 설명하세요.

**풀이 단서**  
caption_q는 한 문장을 요청합니다. 생성한 원문은 다음 문항에서 다시 쓰도록 answer_a에 저장하세요.

In [ ]:
# 여기에 풀이를 작성하세요.
# batch = encode(im, caption_q)
# 각 입력의 shape와 dtype을 확인한 뒤 answer_a를 생성하세요.

**나의 해석**  
출력에서 근거를 한 가지 이상 골라 작성하세요.

✎

### 1-3. 두 설명을 참조 캡션과 사진으로 검수하기

기존 한국어 기록 A와 새 캡션을 참조 문장 5개 및 원본과 비교하세요. 어느 응답이 마릿수를 명시했는지, 사진에서 확인할 수 있는 사실과 “먹이를 기다린다”라는 추론이 어떻게 다른지 쓰세요. 검증한 사실로 한국어 소개 문장을 완성하세요.

**풀이 단서**  
참조 캡션 34438과 37387이 각각 강조하는 정보를 확인하세요. 참조 문장끼리 의견이 갈리면 원본을 다시 보세요.

In [ ]:
# 여기에 풀이를 작성하세요.
# refs = captions.loc[...]
# previous의 A, answer_a, 원본을 같은 자리에서 확인하세요.

**나의 해석**  
출력에서 근거를 한 가지 이상 골라 작성하세요.

✎

### 1-4. 탐지기가 찾은 세 상자와 정답 상자 짝짓기

SSDLite를 원본에 실행하세요. zebra 중 점수 0.50 이상만 남기고 모든 예측×참조 쌍의 IoU를 계산하세요. 각 예측에서 값이 가장 큰 참조가 서로 다른지 확인하고 각 쌍이 IoU 0.50을 넘는지 설명하세요.

**풀이 단서**  
참조는 왼쪽부터, 예측은 모델의 점수순입니다. 두 표의 첫 행이 항상 같은 동물이라고 가정하면 안 됩니다.

In [ ]:
# 여기에 풀이를 작성하세요.
# raw_det = detect(im)
# 행은 예측 P, 열은 참조 GT인 겹침 표를 만드세요.

**나의 해석**  
출력에서 근거를 한 가지 이상 골라 작성하세요.

✎

## 필수 2 · 질문을 바꾸고 마릿수 응답 검증하기

사진과 모델을 그대로 두고 질문만 바꿉니다. 내용이 맞는 답과 입력란에 바로 저장할 수 있는 답을 구분하세요. 이 구간은 공통 준비 직후부터 실행할 수 있습니다.

### 2-1. 질문을 설계하고 응답 두 개를 따로 채점하기

11760에 대해 대상·요청 작업·출력 형식·추측 제한을 담은 질문을 한국어로 먼저 작성하세요. 재현 비교에는 공통 질문 open_q와 strict_q를 사용해 두 응답을 직접 생성하세요. 기존 기록 B와 함께 형식, 마릿수, 두 조건 동시 충족 여부를 비교하세요.

**풀이 단서**  
마침표와 문장도 형식 검사의 대상입니다. 이 단계에서 raw를 지우거나 숫자로 덮어쓰지 마세요.

In [ ]:
# 여기에 풀이를 작성하세요.
# animal = photo(11760)
# truth는 개별 주석 수입니다. 두 질문에 대한 ask와 number_check 결과를 모으세요.

**나의 해석**  
출력에서 근거를 한 가지 이상 골라 작성하세요.

✎

### 2-2. 두 질문을 공정하게 비교할 평가 사진 고르기

배포본에 실제 이미지가 있고 얼룩말 군집 주석이 없는 사진만 대상으로 삼으세요. 예제 11760은 제외하고, 참조 마릿수 1·2·3에서 세 장씩 난수 시드 42로 고르세요. 선택한 사진 ID와 그룹별 수를 기록하세요.

**풀이 단서**  
iscrowd의 합계가 0인 그룹만 포함합니다. 그룹 크기는 주석 행 수입니다. paths에는 직접 열 수 있는 파일 경로만 남겨 두었습니다.

In [ ]:
# 여기에 풀이를 작성하세요.
# counts = zebras.groupby("image_id").iscrowd.agg(["size","sum"])
# sample에는 image_id 인덱스와 reference 열을 만드세요.

**나의 해석**  
출력에서 근거를 한 가지 이상 골라 작성하세요.

✎

### 2-3. 같은 아홉 장에서 프롬프트 두 개 평가하기

사진마다 두 질문을 실행해 총 18개 응답의 원문을 보관하세요. 질문별로 형식 통과, 수 일치, 동시 통과 횟수를 구하고 각각 분모와 함께 보고하세요. 결과가 좋아진 사진과 나빠진 사진을 한 장씩 찾으세요.

**풀이 단서**  
한 행은 사진 한 장과 질문 하나의 조합입니다. 참·거짓 열을 질문별로 더하면 통과 횟수가 됩니다.

In [ ]:
# 여기에 풀이를 작성하세요.
# runs = []
# sample의 각 사진마다 open_q와 strict_q를 ask에 전달하세요.
# trials에 세 검사의 결과를 남기세요.

**나의 해석**  
출력에서 근거를 한 가지 이상 골라 작성하세요.

✎

### 2-4. 오답 사진을 열어 관찰과 원인 추측 구분하기

형식 지정 응답에서 마릿수가 틀린 사례를 우선하고 ID순으로 두 건을 고르세요. 원본·개별 주석·원문을 대조한 뒤 관찰 사실, 가능한 원인, 사람이 고친 설명을 적으세요. 구두점만 없애면 해결되는지도 판단하세요.

**풀이 단서**  
count_ok, both_ok를 오름차순으로 정렬하면 False가 먼저 옵니다. 검토하는 사진에 한해서만 주석 상자를 그리세요.

In [ ]:
# 여기에 풀이를 작성하세요.
# strict_rows에서 두 건을 고르고 각 원본과 painted 결과를 비교하세요.

**나의 해석**  
출력에서 근거를 한 가지 이상 골라 작성하세요.

✎

## 과제 1 · 표지판의 위치와 문구를 함께 제공하기

도로 사진 15440을 지도 서비스에 등록합니다. 서비스에는 원본 기준 상자 좌표와 **사진에서 실제로 읽은 영문**이 필요합니다. 좌표와 글자를 각각 어떤 방법으로 확인할지 정하세요. 공통 준비 직후부터 독립적으로 진행할 수 있습니다.

### 3-1. 실제 탐지 상자로 표지판 위치 찾기

도로 사진에서 탐지기를 실행해 점수 0.50 이상인 stop sign 중 최고 점수를 고르세요. 원본과 정답 상자를 대조해 IoU를 구하고 기존 설명 C만으로는 제공하기 어려운 정보를 설명하세요.

**풀이 단서**  
클래스와 점수 조건을 먼저 적용합니다. 예측 후보를 선택할 때 정답 상자는 쓰지 않습니다. IoU를 잴 때 참조합니다.

In [ ]:
# 여기에 풀이를 작성하세요.
# street = photo(15440)
# best, pred_box, sign_ref, location_iou를 구하세요.

**나의 해석**  
출력에서 근거를 한 가지 이상 골라 작성하세요.

✎

### 3-2. 같은 판독 질문으로 네 가지 입력 범위 비교하기

실제 탐지 영역, 그 너비·높이의 10%씩을 사방에 더한 영역, 정답 주석 영역을 잘라 표시하세요. 전체 사진을 포함한 네 입력에 동일한 영문 판독 질문을 던지고 원문을 기록하세요. 실제 서비스에서 쓸 수 있는 영역과 사후 검증용 영역을 구분하세요.

**풀이 단서**  
상자의 오른쪽 아래 좌표는 잘라내기 범위의 끝입니다. crop 결과의 너비는 오른쪽−왼쪽으로 확인합니다. 질문에 정답 단어를 넣지 마세요.

In [ ]:
# 여기에 풀이를 작성하세요.
# pred_int, pred_crop = crop_box(...)
# dx = 예측 너비 × 0.10, dy = 예측 높이 × 0.10
# answers에 네 원문을 보관하세요.

**나의 해석**  
출력에서 근거를 한 가지 이상 골라 작성하세요.

✎

### 3-3. 읽을 글자가 없는 입력에서도 검증하기

예측 crop과 같은 크기의 균일한 회색 이미지를 생성하세요. 네 사진 입력과 회색 대조 입력에서 같은 질문의 답을 모아 뜻이 맞는지와 지정된 형식까지 일치하는지를 따로 채점하세요. 기존 D와도 대조하세요.

**풀이 단서**  
보이는 표지판에는 STOP, 글자를 모두 없앤 대조 입력에는 UNKNOWN을 기대합니다. 뜻 검사에 허용할 표기를 명시하고 원문은 남겨 두세요.

In [ ]:
# 여기에 풀이를 작성하세요.
# blank = Image.new("RGB", pred_crop.size, ...)
# checks에 원문, 기대값, 내용 일치, 엄격 일치를 구분하세요.

**나의 해석**  
출력에서 근거를 한 가지 이상 골라 작성하세요.

✎

### 3-4. 검수 가능한 서비스 레코드 완성하기

예측 영역에서 영문 단어를 못 얻으면 예측 상자의 여백 버전으로 한 번 재시도하세요. 기권 응답은 빈 값, 대문자 한 단어 뒤의 마침표는 제거해 저장하세요. 표지판 위치·실제 판독 범위·응답 원문·정리한 문구를 남기고, 위치와 문구를 따로 검수하세요.

**풀이 단서**  
정답 상자는 검수에만 사용합니다. 예측이 실패하면 사람의 정답을 복사해 넣지 말고 결과가 왜 보류되는지 기록합니다.

In [ ]:
# 여기에 풀이를 작성하세요.
# word_for_field에 보류 응답과 영문 한 단어 처리 규칙을 구현하세요.
# payload에는 원래 탐지 상자와 OCR 입력 상자를 각각 남기세요.

**나의 해석**  
출력에서 근거를 한 가지 이상 골라 작성하세요.

✎

## 실습 마무리

1. **어떤 문제가 있었나요?** 장면 소개, 숫자 입력, 표지판 등록 중 둘을 골라, 자연스러운 한 문장만으로 채우기 어려운 필드를 쓰세요.
2. **어떻게 개선했나요?** 질문을 바꾼 실험과 이미지 범위를 바꾼 실험에서 각각 무엇을 고정하고 무엇을 바꿨는지 설명하세요.
3. **무엇으로 판단했나요?** 사진 ID, 주석, 응답 원문, 입력 범위, 수치 중 세 가지 이상을 사용해 실제 개선된 점과 여전히 사람이 확인할 점을 적으세요.